In [ ]:
!pip install segmentation-models-pytorch albumentations torch torchvision matplotlib timm torchmetrics tqdm > /dev/null 2>&1

In [ ]:
!pip install --upgrade segmentation-models-pytorch > /dev/null 2>&1

In [ ]:
from google.colab import drive
import os
import torch
import pandas as pd
import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

In [ ]:
# Test Dataset path located in Google Drive
drive.mount('/content/drive')
GDRIVE_PATH = "/content/drive/MyDrive/finalyearproject/"
DATASET_DIR = os.path.join(GDRIVE_PATH, "deepglobe")

metadata_path = os.path.join(DATASET_DIR, "metadata.csv")
metadata_df = pd.read_csv(metadata_path)
test_df = metadata_df[metadata_df['split'] == 'test']

class_dict_path = os.path.join(DATASET_DIR, "class_dict.csv")
class_dict_df = pd.read_csv(class_dict_path)

In [ ]:
class TestDataset(Dataset):
    def __init__(self, metadata, dataset_dir, transform=None):
        self.metadata = metadata
        self.dataset_dir = dataset_dir
        self.transform = transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        img_path = os.path.join(self.dataset_dir, str(row['sat_image_path']))
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            transformed = self.transform(image=image)
            image = transformed["image"]

        return image, img_path


In [ ]:
transform = A.Compose([
    A.PadIfNeeded(min_height=2464, min_width=2464, fill=(0,0,0), fill_mask=6), #ensure size is divisible by 16
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), #Image Net Normalization
    ToTensorV2()
])

test_dataset = TestDataset(test_df, DATASET_DIR, transform=transform)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# load models from Google Drive
model_path_15 = os.path.join(GDRIVE_PATH, "save-resnet101-816-may4-15val/best_model_epoch_15.pth")
model_path_20 = os.path.join(GDRIVE_PATH, "save-resnet101-816-may19-20val/model_epoch_at_2025-05-19 22:23:53_15.pth")
model_path_25 = os.path.join(GDRIVE_PATH, "save-resnet101-816-may4-25val/best_model_epoch_14.pth")
model_path_30 = os.path.join(GDRIVE_PATH, "save-resnet101-816-may4-30val/model_epoch_at_2025-05-04 19:54:40_15.pth")


model_15 = smp.DeepLabV3Plus(encoder_name='resnet101', classes=len(class_dict_df), activation=None).cuda()
model_15.load_state_dict(torch.load(model_path_15))
model_15.eval()

model_20 = smp.DeepLabV3Plus(encoder_name='resnet101', classes=len(class_dict_df), activation=None).cuda()
model_20.load_state_dict(torch.load(model_path_20))
model_20.eval()

model_25 = smp.DeepLabV3Plus(encoder_name='resnet101', classes=len(class_dict_df), activation=None).cuda()
model_25.load_state_dict(torch.load(model_path_25))
model_25.eval()

model_30 = smp.DeepLabV3Plus(encoder_name='resnet101', classes=len(class_dict_df), activation=None).cuda()
model_30.load_state_dict(torch.load(model_path_30))
model_30.eval()

print(f"""Loaded model from {model_path_15}, \n{model_path_20}, \n{model_path_25}, \n{model_path_30}\n""")




In [ ]:
class_dict_path = os.path.join(DATASET_DIR, "class_dict.csv")
class_dict_df = pd.read_csv(class_dict_path)

class_colors = {
    i: (row['r'], row['g'], row['b'])
    for i, row in class_dict_df.iterrows()
}

print("Class Color Mapping:", class_colors)

In [ ]:
def decode_segmentation(mask):
    return torch.argmax(mask, dim=0).cpu().numpy()

def colorize_mask(mask, class_colors):
    """Convert class index mask to an RGB image using predefined colors."""
    h, w = mask.shape
    color_mask = np.zeros((h, w, 3), dtype=np.uint8)

    for class_idx, color in class_colors.items():
        color_mask[mask == class_idx] = color

    return color_mask

In [ ]:
for i, (image, img_path) in enumerate(test_dataloader):
    image = image.cuda()
    with torch.no_grad():


        output_15 = model_15(image)
        output_20 = model_20(image)
        output_25 = model_25(image)
        output_30 = model_30(image)

        predicted_mask_15 = decode_segmentation(output_15.squeeze(0))
        predicted_mask_20 = decode_segmentation(output_20.squeeze(0))
        predicted_mask_25 = decode_segmentation(output_25.squeeze(0))
        predicted_mask_30 = decode_segmentation(output_30.squeeze(0))


    colored_mask_15 = colorize_mask(predicted_mask_15, class_colors)
    colored_mask_20 = colorize_mask(predicted_mask_20, class_colors)
    colored_mask_25 = colorize_mask(predicted_mask_25, class_colors)
    colored_mask_30 = colorize_mask(predicted_mask_30, class_colors)


    orig_img = cv2.imread(img_path[0])
    orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(25, 10))
    plt.subplot(1, 5, 1)
    plt.imshow(orig_img)
    plt.title(f"Original Image {i+1}")


    plt.subplot(1, 5, 2)
    plt.imshow(colored_mask_15)
    plt.title(f"Predicted Segmentation 15 validation {i+1}")

    plt.subplot(1, 5, 3)
    plt.imshow(colored_mask_20)
    plt.title(f"Predicted Segmentation 20 validation {i+1}")

    plt.subplot(1, 5, 4)
    plt.imshow(colored_mask_25)
    plt.title(f"Predicted Segmentation 25 validation {i+1}")

    plt.subplot(1, 5, 5)
    plt.imshow(colored_mask_30)
    plt.title(f"Predicted Segmentation 30 validation {i+1}")

    plt.show()
